In [3]:
import os
import pathlib

import json
from tqdm.auto import tqdm

import requests

In [5]:
with open(target_db / 'root_data.json') as f:
    root_data = json.load(f)

In [8]:
root_data_filtered = []
for crag in root_data:
    if crag['bought']:
        root_data_filtered.append(crag)

# Crawl API

In [4]:
with open('token.txt') as f:
    token = f.read().strip()

In [5]:
target_db = pathlib.Path('./guidebook-app/public/database/')

In [6]:
header = {'authorization': f"Token {token}"}

In [7]:
root_url = "https://www.versantesud.it/api/keinwunder/climbing/sport/"

In [8]:
response = requests.get(root_url ,headers=header)

In [9]:
response.ok

True

In [7]:
root_data = response.json()

with open(target_db / 'root_data.json', 'w') as f:
    json.dump(root_data, f)

In [8]:
root_data_filtered = []
for crag in root_data:
    if crag['bought']:
        root_data_filtered.append(crag)

In [9]:
with open(target_db / 'root_data_bought.json', 'w') as f:
    json.dump(root_data_filtered, f)

In [11]:
# Purchased guides
url = "https://www.versantesud.it/api/keinwunder/climbing/purchased/guide/"
response = requests.get(url ,headers=header)
assert response.ok
purchased_guide_data = response.json()
with open(target_db / 'purchased_guides.json', 'w') as f:
    json.dump(purchased_guide_data, f)

In [13]:
with open(target_db / 'purchased_guides.json') as f:
    purchased_guide_data = json.load(f)

In [14]:
# Book cover images
for guide in tqdm(purchased_guide_data):
    guide_id = f"{guide['title']}_{guide['year']}"
    file_name = guide['cover_image'].split('/')[-1]
    
    output_file = target_db / 'images_book_covers' / file_name
    
    if output_file.is_file():
        continue
    
    
    response = requests.get(guide['cover_image'])
    assert response.ok
    
    with open(output_file, 'wb') as f:
        f.write(response.content)
    

  0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
# Download crags
for crag in tqdm(root_data_filtered):
    if not crag['bought']:
        continue
    
    assert crag['url'].startswith(root_url)
    assert crag['url'].endswith('/')
    crag_id = crag['url'][len(root_url):-1]
    
    output_path = target_db / f'crags/{crag_id}.json'
    
    if output_path.is_file():
        continue
    
    
    response = requests.get(crag['url'], headers=header)
    assert response.ok
    
    crag_data = response.json()
    
    with open(output_path, 'w') as f:
        json.dump(crag_data, f)
    

  0%|          | 0/375 [00:00<?, ?it/s]

In [17]:
crag_ids_bought = []
for crag in tqdm(root_data):
    if not crag['bought']:
        continue
    
    assert crag['url'].startswith(root_url)
    assert crag['url'].endswith('/')
    crag_id = crag['url'][len(root_url):-1]
    
    crag_ids_bought.append(crag_id)

  0%|          | 0/1493 [00:00<?, ?it/s]

In [18]:
len(crag_ids_bought)

375

In [19]:
# Download approaches and sectors (list of sectors) for each crag
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'crags/{crag_id}.json') as f:
        crag_data = json.load(f)
        
        
    assert crag_data['approach_url'] == root_url + crag_id + '/approach/'
    assert crag_data['sectors_url'] == root_url + crag_id + '/sectors/'
    
    output_path_approach = target_db / f'approaches/{crag_id}.json'
    output_path_crag = target_db / f'sectors/{crag_id}.json'
    
        
    if not output_path_approach.is_file():
        response = requests.get(crag_data['approach_url'], headers=header)
        assert response.ok
        approach_data = response.json()
        with open(output_path_approach, 'w') as f:
            json.dump(approach_data, f)
    
    
    if not output_path_crag.is_file():
        response = requests.get(crag_data['sectors_url'], headers=header)
        assert response.ok
        sectors_data = response.json()
        with open(output_path_crag, 'w') as f:
            json.dump(sectors_data, f)

  0%|          | 0/375 [00:00<?, ?it/s]

In [20]:
# download sector instances
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'sectors/{crag_id}.json') as f:
        sector_data = json.load(f)
        
    for sector in sector_data['sectors']:
        sector_id = sector['url'].split('/')[-2]
        assert sector['url'] == root_url + crag_id + '/sector/' + sector_id + '/'
        
        output_path = target_db / f'sector_instance/{crag_id}_{sector_id}.json'
        if output_path.is_file():
            continue
        
        response = requests.get(sector['url'], headers=header)
        assert response.ok
        
        sector_data = response.json()
        with open(output_path, 'w') as f:
            json.dump(sector_data, f)

  0%|          | 0/375 [00:00<?, ?it/s]

In [21]:
# download topo images
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'sectors/{crag_id}.json') as f:
        sector_data = json.load(f)
        
    for sector in sector_data['sectors']:
        sector_id = sector['url'].split('/')[-2]
        
        with open(target_db / f'sector_instance/{crag_id}_{sector_id}.json') as f:
            sector_instance_data = json.load(f)
            
        for image in sector_instance_data['images']:
            image_filename = image['url'].split('/')[-1]
            output_file = target_db / 'images_topos' / image_filename
            
            if output_file.is_file():
                continue
            
            response = requests.get(image['url'])
            assert response.ok
            
            with open(output_file, 'wb') as f:
                f.write(response.content)

  0%|          | 0/375 [00:00<?, ?it/s]

In [22]:
# Download approach images
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'approaches/{crag_id}.json') as f:
        approach_data = json.load(f)
        
    if approach_data['image'] is None:
        print('No approach image for', crag_id)
        continue
    
    image_filename = approach_data['image'].split('/')[-1]
    
    output_file = target_db / f'images_approaches/{image_filename}'
    if output_file.is_file():
        continue
    
    
    response = requests.get(approach_data['image'])
    assert response.ok
    
    with open(output_file, 'wb') as f:
        f.write(response.content)
        

  0%|          | 0/375 [00:00<?, ?it/s]

No approach image for 240
No approach image for 327
No approach image for 277
No approach image for 306


In [30]:
approach_data

{'description': "Da Longarone seguire le indicazioni per Zoldo e l'omonima valle. Dopo il tunnel proseguire fino alla rotonda ed entrare nella piccola frazione di Igne. Nei pressi della chiesa continuare seguendo la strada in salita, fare due tornanti e subito dopo parcheggiare sulla destra nel piccolo spazio disponibile alla fine del paese. Dal piccolo parcheggio una traccia di sentiero porta in 5 minuti alla base della parete già ben visibile dalla macchina.",
 'area_name': 'Cadore',
 'area_description': '',
 'image': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/20/0a/200a0f68-4f0f-4ae8-b240-541f7afd0634/igne_accesso.png__1500x1200_ALIAS-iclimbing_approach_subsampling-2.png',
 'parking_latitude': 46.266819,
 'parking_longitude': 12.277698,
 'track': None,
 'length': None,
 'elevation_gain': None,
 'time': 5}

In [29]:
crag

{'name': 'Igne',
 'latitude': 46.268653,
 'longitude': 12.277783,
 'url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3767/',
 'toggle_favourite_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3767/toggle_favourite/',
 'published': True,
 'histogram': [{'label': '<5', 'count': 0},
  {'label': '5a', 'count': 0},
  {'label': '5b', 'count': 0},
  {'label': '5c', 'count': 0},
  {'label': '6a', 'count': 1},
  {'label': '6b', 'count': 2},
  {'label': '6c', 'count': 6},
  {'label': '7a', 'count': 21},
  {'label': '7b', 'count': 12},
  {'label': '7c', 'count': 13},
  {'label': '8a', 'count': 14},
  {'label': '8b', 'count': 7},
  {'label': '8c', 'count': 3},
  {'label': '9', 'count': 0},
  {'label': '?', 'count': 1}],
 'characteristics': ['5 min', '700 m slm', 'Sud-ovest'],
 'exposition': 'Sud-ovest',
 'expositions': ['Sud-ovest'],
 'altitude': 700,
 'approach_time': 5,
 'family_friendly': True,
 'comfortable_base': True,
 'easy_parking': False,
 'gorgeous': True,


In [26]:
crag_data

{'name': 'Igne',
 'cover_image': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/f5/c4/f5c41778-c5c3-4118-933e-f0f774cb4e9f/immagine_copertina_dolomiti_falesie.jpg__4096x3072_q85_ALIAS-iclimbing_fullscreen_subsampling-2.jpg',
 'cover_characteristics': [{'name': 'Avvicinamento', 'value': '5 min'},
  {'name': 'Altitudine', 'value': '700 m slm'},
  {'name': 'Esposizione', 'value': 'Sud-ovest'}],
 'images': [{'thumbnail_url': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/e1/4e/e14e5897-fa53-421d-b1b1-df58647d2b6d/igne_vista.jpg__15000x1500_q85_ALIAS-iclimbing_thumbnail_subsampling-2.jpg',
   'url': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/e1/4e/e14e5897-fa53-421d-b1b1-df58647d2b6d/igne_vista.jpg__4096x3072_q85_ALIAS-iclimbing_fullscreen_subsampling-2.jpg'}],
 'date_updated': '2026-09-18T18:23:38.464545+02:0

In [27]:
sector_data

{'image': None,
 'sectors': [{'name': 'SENZA NOME',
   'min_difficulty': '6a',
   'max_difficulty': '8c+',
   'route_count': 80,
   'url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3767/sector/6782/'}]}

In [28]:
sector_instance_data

{'name': 'SENZA NOME',
 'approach_time': None,
 'histogram': [{'label': '<5', 'count': 0},
  {'label': '5a', 'count': 0},
  {'label': '5b', 'count': 0},
  {'label': '5c', 'count': 0},
  {'label': '6a', 'count': 1},
  {'label': '6b', 'count': 2},
  {'label': '6c', 'count': 6},
  {'label': '7a', 'count': 21},
  {'label': '7b', 'count': 12},
  {'label': '7c', 'count': 13},
  {'label': '8a', 'count': 14},
  {'label': '8b', 'count': 7},
  {'label': '8c', 'count': 3},
  {'label': '9', 'count': 0},
  {'label': '?', 'count': 1}],
 'images': [{'thumbnail_url': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/4a/fe/4afec51c-23d3-4077-981f-e5f5f55e7c02/igne.png__15000x1500_ALIAS-iclimbing_thumbnail_subsampling-2.png',
   'url': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/4a/fe/4afec51c-23d3-4077-981f-e5f5f55e7c02/igne.png__4096x3072_ALIAS-iclimbing_fullscreen_subsampling-2.png'

# Downsample images

In [23]:
from PIL import Image

In [24]:
target_db = pathlib.Path('./guidebook-app/public/database/')

In [25]:
for cat in ['approaches', 'topos']:
    max_size = 500 if cat == 'approaches' else 800

    path_src = target_db / f'images_{cat}'
    path_dst = target_db / f'images_optimized_{cat}'
    path_dst.mkdir(exist_ok=True)

    for img_path in tqdm(list(path_src.iterdir())):
        assert img_path.name.endswith('.png') or img_path.name.endswith('.jpg')
        
        im = Image.open(img_path).convert('RGB')
        im.thumbnail((max_size, max_size), )
        
        
        
        img_path_dst = path_dst / (img_path.name + '.jpg')
        im.save(img_path_dst, 'JPEG', optimize=True, quality=75)
        
        

  0%|          | 0/352 [00:00<?, ?it/s]

  0%|          | 0/910 [00:00<?, ?it/s]

# Parse Isar

In [2]:
with open("offline_spots_v1.isar", "rb") as f:
    data = f.read()

In [42]:
i = 0
j = 0
datas = []
while i != -1:
    i = data.find(b'{', j)
    j = data.find(b'\x00', i)
    
    data_str = data[i:j-1]
    
    data_str = ''.join(chr(x) for x in data_str if x in range(128))
    
    if not data_str:
        continue
    
    data_parsed = json.loads(data_str)
    print(data_parsed)
    print('===')
    
    datas.append(data_parsed)

{'name': 'OfflineSpot', 'embedded': False, 'properties': [{'name': 'apiVersion', 'type': 'String', 'target': None}, {'name': 'approachImagePath', 'type': 'String', 'target': None}, {'name': 'approachJson', 'type': 'String', 'target': None}, {'name': 'boulderDetailsJson', 'type': 'String', 'target': None}, {'name': 'coverImagePath', 'type': 'String', 'target': None}, {'name': 'discipline', 'type': 'Byte', 'target': None}, {'name': 'displayName', 'type': 'String', 'target': None}, {'name': 'downloadedAt', 'type': 'Long', 'target': None}, {'name': 'formattedSize', 'type': 'String', 'target': None}, {'name': 'galleryImagePaths', 'type': 'StringList', 'target': None}, {'name': 'guideCoverImagePath', 'type': 'String', 'target': None}, {'name': 'hasTileRegions', 'type': 'Bool', 'target': None}, {'name': 'isValid', 'type': 'Bool', 'target': None}, {'name': 'latitude', 'type': 'Double', 'target': None}, {'name': 'longitude', 'type': 'Double', 'target': None}, {'name': 'mapSnapshotPath', 'type':

In [53]:
datas[-1]

{'name': 'FALESIA DELLORSO',
 'histogram': [{'label': '<5', 'count': 0},
  {'label': '5a', 'count': 0},
  {'label': '5b', 'count': 0},
  {'label': '5c', 'count': 0},
  {'label': '6a', 'count': 2},
  {'label': '6b', 'count': 2},
  {'label': '6c', 'count': 2},
  {'label': '7a', 'count': 2},
  {'label': '7b', 'count': 1},
  {'label': '7c', 'count': 1},
  {'label': '8a', 'count': 2},
  {'label': '8b', 'count': 0},
  {'label': '8c', 'count': 0},
  {'label': '9', 'count': 0},
  {'label': '?', 'count': 2}],
 'characteristics': ['Base confortevole',
  'Disponibilit parcheggio',
  'Splendido',
  'Ottima chiodatura',
  'Tranquillo / non affollato',
  'Calcare'],
 'cover_image': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/ac/76/ac76ae84-1f07-43ad-a614-0c9fad60a177/immagine_cop_trentoevalsuganafalesie.jpg__4096x3072_q85_ALIAS-iclimbing_fullscreen_subsampling-2.jpg',
 'cover_characteristics': [{'name': 'Avvicinamento', 'value': '15 min'

In [52]:
json.loads(list(datas[-3].values())[0])['routes']

[{'number': 1,
  'name': 'SENZA NOME',
  'beautiful': True,
  'pitches': [{'name': 'L1',
    'length': None,
    'comment': '',
    'difficulty_grade': None,
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55344/pitch/59206/toggle_done/'}]},
 {'number': 2,
  'name': 'PENTECOSTE',
  'beautiful': False,
  'pitches': [{'name': 'L1',
    'length': 30,
    'comment': 'Duro a met',
    'difficulty_grade': '7b',
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55345/pitch/59207/toggle_done/'}]},
 {'number': 3,
  'name': 'SENZA NOME',
  'beautiful': False,
  'pitches': [{'name': 'L1',
    'length': 30,
    'comment': 'Continuit',
    'difficulty_grade': '6c+',
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55346/pitch/59208/toggle_done/'}]},
 {'number': 4,
  'name': 'SENZA